<hr style="border:2px solid #0281c9"> </hr>

<img align="left" alt="ESO Logo" src="http://archive.eso.org/i/esologo.png">  

<div align="center">
  <h1 style="color: #0281c9; font-weight: bold;">ESO Science Archive</h1> 
  <h2 style="color: #0281c9; font-weight: bold;">Jupyter Notebooks</h2>
</div>

<hr style="border:2px solid #0281c9"> </hr>

# **Query Ancillary Files for ESO Phase 3 Products**

This notebook shows how to find files that belong to an ESO Phase 3 science product using `astroquery.eso` and TAP/ADQL.

A Phase 3 product always has one science file and may also have ancillary files. Ancillary files can include previews, weight maps, masks, white-light images, telluric spectra, tar archives, or other files produced alongside the science product. These files are not listed as independent science products in `ivoa.ObsCore`; instead, they are listed in the `phase3v2.product_files` table.

The workflow presented in this example is:

- inspect the metadata for `phase3v2.product_files`
- list all files belonging to one product ID
- filter ancillary files by ESO category and original filename
- join `phase3v2.product_files` to `ivoa.ObsCore` to search ancillary files across a collection
- estimate the download volume and list direct access URLs for a selected file type

The examples are adapted from the ESO Science Archive programmatic HOWTO on ancillary files, but use `astroquery.eso.Eso.query_tap()` throughout.

<hr style="border:2px solid #0281c9"> </hr>

## **Importing and basic usage of astroquery.eso**

In [1]:
import astroquery
from astroquery.eso import Eso

print(f"astroquery version: {astroquery.__version__}")

eso = Eso()

astroquery version: 0.4.12.dev505+gf2a77a615.d20260427


## **Inspect the `phase3v2.product_files` table**

`phase3v2.product_files` describes the file components of Phase 3 products. Useful columns of note here are:

- `product_id`: the Phase 3 product identifier, matching `ivoa.ObsCore.dp_id`
- `archive_id`: the identifier of the individual file; for the science file it is usually the same as `product_id`, while ancillary files have their own identifiers
- `eso_category`: the ESO science-data-product category, for example `SCIENCE.SPECTRUM` or `ANCILLARY.PREVIEW`
- `original_filename`: the filename before ingestion into the archive, often useful for filtering files within one release
- `access_url` and `access_estsize`: the direct download URL and estimated size of each file

Querying `TAP_SCHEMA.columns` keeps those names and meanings discoverable from the service itself.

In [2]:
columns_query = """
                SELECT column_name, ucd, description
                FROM TAP_SCHEMA.columns
                WHERE table_name = 'phase3v2.product_files'
                ORDER BY column_name
                """

product_file_columns = eso.query_tap(columns_query)
product_file_columns

column_name,ucd,description
object,object,object
access_estsize,phys.size;meta.file,"Estimated size of the downloaded file in KBytes. It is only ""estimated"" as in general, FITS headers can be patched at download time, making the file size varying with time."
access_url,meta.ref.url,The download link of the individual archive_id file.
archive_id,meta.id;meta.file;meta.main,"ESO identifier of a file belonging to the product: either a science file (eso_category: ""SCIENCE.*"") or an ancillary file (eso_category: ""ANCILLARY.*""). If a science file, its id is the same as the id of the product (product_id=archive_id). Ancillary files are not listed in the ivoa.ObsCore table."
eso_category,meta.code.class;meta.file,"ESO file category; It starts with ""SCIENCE."" or ""ANCILLARY."" followed by one or more dot-separated and uppercased tokens describing the file at hand. For the full list of categories, please refer to the ESO Science Data Product standard available at: https://www.eso.org/sci/observing/phase3.html"
extension,meta.id.part;meta.file,"File name extension (upper case) of the product file, e.g.: FITS, PNG, TAR, FZ, etc."
internal_file_id,meta.id;meta.file,"Internal file identification number, useful in combination with the provenance table."
original_filename,meta.id,"Original name of the product file before ingestion in the ESO archive. It may provide useful hints on the file content (usually described in the release description of the product, available at: https://archive.eso.org/wdb/wdb/adp/phase3_main/query?dp_id=here_the_product_id). It is not an identifier, as in general, there could be multiple files with the same original_filename but different archive_ids."
product_id,meta.id;meta.dataset;meta.main,"ESO identifier of a published phase 3 product. A product is composed of a science file, and optionally of a number of ancillary files. Same as the dp_id field in ivoa.ObCore."


## **Example 1: list all files belonging to one product**

Start from a single product ID. The query below returns the science file and any ancillary files associated with that product. Splitting the output into selected columns makes the result easier to inspect in a notebook than a full `SELECT *`. In this case we check the product ID (`dp_id`) `ADP.2024-03-19T05:58:04.407`, which we can check first in `eso.query_survey()` - alternatively, we could have found this product ID from a previous query of `eso.query_survey()` and then passed the output `dp_id` to the `phase3v2.product_files` TAP query below.

In [3]:
product_id = "ADP.2024-03-19T05:58:04.407"

table = eso.query_surveys(column_filters={"dp_id": product_id})
table

target_name,s_ra,s_dec,dp_id,proposal_id,abmaglim,access_estsize,access_format,access_url,bib_reference,calib_level,dataproduct_subtype,dataproduct_type,em_max,em_min,em_res_power,em_xel,facility_name,filter,gal_lat,gal_lon,instrument_name,is_solar,last_mod_date,multi_ob,n_obs,o_calib_status,o_ucd,obs_collection,obs_creator_did,obs_creator_name,obs_id,obs_publisher_did,obs_release_date,obs_title,obstech,p3orig,pol_states,pol_xel,preview_html,publication_date,release_description,s_fov,s_pixel_scale,s_region,s_resolution,s_xel1,s_xel2,snr,strehl,t_exptime,t_max,t_min,t_resolution,t_xel
,deg,deg,,,mag,kbyte,,,,,,,m,m,,,,,deg,deg,,,,,,,,,,,,,,,,,,,,,,deg,arcsec,,arcsec,,,,,s,d,d,s,
object,float64,float64,object,object,float64,int64,object,object,object,int32,object,object,float64,float64,float64,int64,object,object,float64,float64,object,int16,object,object,int32,object,object,object,object,object,object,object,object,object,object,object,object,int64,object,object,object,float64,float64,object,float64,int64,int64,float64,float64,float64,float64,float64,float64,int64
SKY_for_NGC4289,185.224495,3.712554,ADP.2024-03-19T05:58:04.407,110.24AS.002,23.168,2968672,application/x-votable+xml;content=datalink,http://archive.eso.org/datalink/links?ID=ivo://eso.org/ID?ADP.2024-03-19T05:58:04.407,,2,ifs,cube,9.35014e-07,4.75014e-07,3026.7861328125,3681,ESO-VLT-U4,,65.473896,284.305656,MUSE,0,2024-03-19T06:07:35.937Z,S,2,absolute,,MUSE,ivo://eso.org/origfile?MU_SCBY_3442284_2024-03-07T08:09:46.977_WFM-NOAO-N_SKY.fits,"VAN DE SANDE, JESSE",3442284,ivo://eso.org/ID?ADP.2024-03-19T05:58:04.407,2025-03-07T08:45:40Z,,IFU,IDP,,--,https://archive.eso.org/dataset/ADP.2024-03-19T05:58:04.407,2024-03-19T06:03:22Z,http://www.eso.org/rm/api/v1/public/releaseDescriptions/78,0.02494516138,0.2,POLYGON J2000 185.233319 3.703777 185.215616 3.703777 185.215615 3.721388 185.233319 3.721388,0.836,318,318,--,--,301.474,60376.36076165,60376.34012705,1782.82944,--


In [4]:
files_query = f"""
                SELECT
                    archive_id,
                    extension,
                    eso_category,
                    original_filename
                FROM phase3v2.product_files
                WHERE product_id = '{product_id}'
                ORDER BY eso_category, archive_id
                """

product_files = eso.query_tap(files_query)
product_files

archive_id,extension,eso_category,original_filename
object,object,object,object
ADP.2024-03-19T05:58:04.408,FITS,ANCILLARY.IMAGE.WHITELIGHT,MU_SIMY_3442284_2024-03-07T08:09:46.977_WFM-NOAO-N_SKY.fits
ADP.2024-03-19T05:58:04.410,PNG,ANCILLARY.PREVIEW,r.MUSE.2024-03-07T08:09:46.977_tpl.png
ADP.2024-03-19T05:58:04.411,PNG,ANCILLARY.PREVIEW,r.MUSE.2024-03-07T08:09:46.977_pst.png
ADP.2024-03-19T05:58:04.412,PNG,ANCILLARY.PREVIEW,r.MUSE.2024-03-07T08:36:46.807_pst.png
ADP.2024-03-19T05:58:04.409,LOG,ANCILLARY.README,r.MUSE.2024-03-07T08:09:46.977_tpl.log
ADP.2024-03-19T05:58:04.407,FITS,SCIENCE.CUBE.IFS,MU_SCBY_3442284_2024-03-07T08:09:46.977_WFM-NOAO-N_SKY.fits


The access columns are the ones to use when preparing a download list. `access_estsize` is reported in kilobytes, so the query below also converts it to megabytes for a quick size check.

In [5]:
access_query = f"""
                SELECT
                    archive_id,
                    eso_category,
                    access_url,
                    access_estsize / 1000.0 AS access_estsize_mb
                FROM phase3v2.product_files
                WHERE product_id = '{product_id}'
                ORDER BY eso_category, archive_id
                """

product_access = eso.query_tap(access_query)
product_access

archive_id,eso_category,access_url,access_estsize_mb
object,object,object,float64
ADP.2024-03-19T05:58:04.408,ANCILLARY.IMAGE.WHITELIGHT,https://dataportal.eso.org/dataPortal/file/ADP.2024-03-19T05:58:04.408,0.529
ADP.2024-03-19T05:58:04.410,ANCILLARY.PREVIEW,https://dataportal.eso.org/dataPortal/file/ADP.2024-03-19T05:58:04.410,0.34
ADP.2024-03-19T05:58:04.411,ANCILLARY.PREVIEW,https://dataportal.eso.org/dataPortal/file/ADP.2024-03-19T05:58:04.411,0.341
ADP.2024-03-19T05:58:04.412,ANCILLARY.PREVIEW,https://dataportal.eso.org/dataPortal/file/ADP.2024-03-19T05:58:04.412,0.335
ADP.2024-03-19T05:58:04.409,ANCILLARY.README,https://dataportal.eso.org/dataPortal/file/ADP.2024-03-19T05:58:04.409,0.241
ADP.2024-03-19T05:58:04.407,SCIENCE.CUBE.IFS,https://dataportal.eso.org/dataPortal/file/ADP.2024-03-19T05:58:04.407,2968.672


## **Example 2: filtering ESPRESSO ancillary spectra**

The original_filename column is often useful when a collection includes multiple ancillary files under the same ESO category. In this ESPRESSO example, a stacked one-dimensional spectrum is accompanied by ancillary spectra corresponding to the individual exposures that contributed to the stack.

In [6]:
espresso_product_id = "ADP.2021-04-12T12:27:33.887"

table = eso.query_surveys(column_filters={"dp_id": espresso_product_id})
table

target_name,s_ra,s_dec,dp_id,proposal_id,abmaglim,access_estsize,access_format,access_url,bib_reference,calib_level,dataproduct_subtype,dataproduct_type,em_max,em_min,em_res_power,em_xel,facility_name,filter,gal_lat,gal_lon,instrument_name,is_solar,last_mod_date,multi_ob,n_obs,o_calib_status,o_ucd,obs_collection,obs_creator_did,obs_creator_name,obs_id,obs_publisher_did,obs_release_date,obs_title,obstech,p3orig,pol_states,pol_xel,preview_html,publication_date,release_description,s_fov,s_pixel_scale,s_region,s_resolution,s_xel1,s_xel2,snr,strehl,t_exptime,t_max,t_min,t_resolution,t_xel
,deg,deg,,,mag,kbyte,,,,,,,m,m,,,,,deg,deg,,,,,,,,,,,,,,,,,,,,,,deg,arcsec,,arcsec,,,,,s,d,d,s,
object,float64,float64,object,object,float64,int64,object,object,object,int32,object,object,float64,float64,float64,int64,object,object,float64,float64,object,int16,object,object,int32,object,object,object,object,object,object,object,object,object,object,object,object,int64,object,object,object,float64,float64,object,float64,int64,int64,float64,float64,float64,float64,float64,float64,int64
WASP-76,26.632804,2.70069,ADP.2021-04-12T12:27:33.887,1102.C-0744(C),--,17812,application/x-votable+xml;content=datalink,http://archive.eso.org/datalink/links?ID=ivo://eso.org/ID?ADP.2021-04-12T12:27:33.887,,2,flux-calibrated,spectrum,7.89997e-07,3.7719999999999996e-07,140000.0,443262,ESO-VLT-U3,,-57.346687,149.084662,ESPRESSO,0,2026-02-17T12:25:11.003Z,S,5,absolute,,ESPRESSO,ivo://eso.org/origfile?ES_SOBF_2174806_2018-09-03T09:11:43.369_HR_2x1_U3.fits,"PEPE, FRANCESCO",2174806,ivo://eso.org/ID?ADP.2021-04-12T12:27:33.887,2021-11-25T11:13:39.583Z,WASP-76_ES_SOBF_2174806_2018-09-03T09:11:43.369_HR_2x1_U3,ECHELLE,IDP,,--,https://archive.eso.org/dataset/ADP.2021-04-12T12:27:33.887,2021-04-29T07:11:38Z,http://www.eso.org/rm/api/v1/public/releaseDescriptions/176,0.00027777,--,POSITION J2000 26.632803999999993 2.70069,--,--,--,203.1,--,3000.0,58364.42091863,58364.38314084,3264.001056,--


In [7]:
espresso_files_query = f"""
                        SELECT
                            archive_id,
                            original_filename,
                            eso_category
                        FROM phase3v2.product_files
                        WHERE product_id = '{espresso_product_id}'
                        ORDER BY eso_category, original_filename
                        """

espresso_files = eso.query_tap(espresso_files_query)
espresso_files[:3]

archive_id,original_filename,eso_category
object,object,object
ADP.2021-04-12T12:27:33.903,ES_SOBF_2174806_2018-09-03T09:11:43.369_HR_2x1_U3.tar,ANCILLARY.2DECHELLE.TAR
ADP.2021-04-12T12:27:33.898,ES_SCCA_2174806_2018-09-03T09:11:43.369_HR_2x1_U3.fits,ANCILLARY.CCF
ADP.2021-04-12T12:27:33.899,ES_SCCA_2174806_2018-09-03T09:22:49.937_HR_2x1_U3.fits,ANCILLARY.CCF


As of 2024, the ESPRESSO pipeline stores measured radial velocities in the headers of the individual (non-stacked) 1D spectra, and not in the stacked product itself. To recover these measurements, you therefore need to access the relevant `ANCILLARY.SPECTRUM` files associated with the stacked product.

These ancillary spectra include:
- Object (fiber A) spectra
- Sky or Fabry–Perot (fiber B) spectra

To isolate the object spectra (fiber A), you can use the original_filename column:
- filenames starting with `ES_SFLA` → object (fiber A)  
- filenames starting with `ES_SFLAB` → sky / Fabry–Perot (fiber B)

Note that filename conventions can vary between releases, so they should always be verified against the corresponding data-release documentation.

### 🔎 Where does this information come from?

Details on filename conventions and data structure are provided in the ESPRESSO data-release documentation, accessible via the product page. For example:

- Product entry: https://archive.eso.org/dataset/ADP.2021-04-12T12:27:33.887 
- Release description: https://www.eso.org/rm/api/v1/public/releaseDescriptions/176  

These resources describe the naming scheme and content of ancillary files, enabling targeted queries such as the one shown below.

In [8]:
espresso_fiber_a_query = f"""
                            SELECT
                                archive_id,
                                original_filename,
                                access_url
                            FROM phase3v2.product_files
                            WHERE product_id = '{espresso_product_id}'
                            AND eso_category = 'ANCILLARY.SPECTRUM'
                            AND original_filename LIKE 'ES_SFLA%'
                            ORDER BY original_filename
                            """

espresso_fiber_a_spectra = eso.query_tap(espresso_fiber_a_query)
espresso_fiber_a_spectra

archive_id,original_filename,access_url
object,object,object
ADP.2021-04-12T12:27:33.888,ES_SFLA_2174806_2018-09-03T09:11:43.369_HR_2x1_U3.fits,https://dataportal.eso.org/dataPortal/file/ADP.2021-04-12T12:27:33.888
ADP.2021-04-12T12:27:33.889,ES_SFLA_2174806_2018-09-03T09:22:49.937_HR_2x1_U3.fits,https://dataportal.eso.org/dataPortal/file/ADP.2021-04-12T12:27:33.889
ADP.2021-04-12T12:27:33.890,ES_SFLA_2174806_2018-09-03T09:33:56.370_HR_2x1_U3.fits,https://dataportal.eso.org/dataPortal/file/ADP.2021-04-12T12:27:33.890
ADP.2021-04-12T12:27:33.891,ES_SFLA_2174806_2018-09-03T09:45:02.953_HR_2x1_U3.fits,https://dataportal.eso.org/dataPortal/file/ADP.2021-04-12T12:27:33.891
ADP.2021-04-12T12:27:33.892,ES_SFLA_2174806_2018-09-03T09:56:07.370_HR_2x1_U3.fits,https://dataportal.eso.org/dataPortal/file/ADP.2021-04-12T12:27:33.892


## **Example 3: find MUSE preview files across the archive**

To search ancillary files for an entire collection, join `phase3v2.product_files` to `ivoa.ObsCore`. The join key is `phase3v2.product_files.product_id = ivoa.ObsCore.dp_id`.

The next query asks for a few MUSE preview files. `TOP 3` keeps the notebook output small while still showing the returned columns.

In [9]:
muse_preview_query = """
                    SELECT TOP 3
                        product_files.product_id,
                        product_files.archive_id,
                        product_files.original_filename,
                        product_files.access_url
                    FROM phase3v2.product_files AS product_files
                    INNER JOIN ivoa.ObsCore AS obscore
                        ON obscore.dp_id = product_files.product_id
                    WHERE obscore.obs_collection = 'MUSE'
                    AND product_files.eso_category = 'ANCILLARY.PREVIEW'
                    """

muse_previews = eso.query_tap(muse_preview_query)
muse_previews

product_id,archive_id,original_filename,access_url
object,object,object,object
ADP.2016-06-01T14:07:30.134,ADP.2016-06-01T14:07:30.137,r.MUSE.2015-01-12T00:54:04.379_tpl.png,https://dataportal.eso.org/dataPortal/file/ADP.2016-06-01T14:07:30.137
ADP.2016-06-01T14:07:30.134,ADP.2016-06-01T14:07:30.138,r.MUSE.2015-01-12T00:54:04.379_pst.png,https://dataportal.eso.org/dataPortal/file/ADP.2016-06-01T14:07:30.138
ADP.2016-06-01T14:07:30.134,ADP.2016-06-01T14:07:30.139,r.MUSE.2015-01-12T01:00:44.954_pst.png,https://dataportal.eso.org/dataPortal/file/ADP.2016-06-01T14:07:30.139


The TAP service also publishes relationship metadata for joins. The query below asks the TAP schema which column in `phase3v2.product_files` points to `ivoa.ObsCore`. This is a useful pattern when you want to verify table relationships rather than rely on hard-coded assumptions.

In [10]:
join_metadata_query = """
                        SELECT
                            keys.from_table,
                            key_columns.from_column,
                            keys.target_table,
                            key_columns.target_column
                        FROM TAP_SCHEMA.keys AS keys
                        INNER JOIN TAP_SCHEMA.key_columns AS key_columns
                            ON keys.key_id = key_columns.key_id
                        WHERE keys.from_table = 'phase3v2.product_files'
                        AND keys.target_table = 'ivoa.ObsCore'
                        """

join_metadata = eso.query_tap(join_metadata_query)
join_metadata

from_table,from_column,target_table,target_column
object,object,object,object
phase3v2.product_files,product_id,ivoa.ObsCore,dp_id


## **Estimate the size of a bulk ancillary download**

Before downloading many ancillary files, query the total number of files and their estimated size.

In [11]:
muse_preview_size_query = """
                            SELECT
                                COUNT(*) AS num_files,
                                SUM(product_files.access_estsize) / 1000.0 AS size_mb,
                                SUM(product_files.access_estsize) / 1000.0 / 1000.0 AS size_gb
                            FROM phase3v2.product_files AS product_files
                            INNER JOIN ivoa.ObsCore AS obscore
                                ON obscore.dp_id = product_files.product_id
                            WHERE obscore.obs_collection = 'MUSE'
                            AND product_files.eso_category = 'ANCILLARY.PREVIEW'
                            """

muse_preview_size = eso.query_tap(muse_preview_size_query)
muse_preview_size

num_files,size_mb,size_gb
int32,float64,float64
73637,19078.038,19.078038


## **Download an ancillary files**

The `dp_id` (`product_id`) is used to download the science (product) file via `eso.retrieve_data()`. To download ancillary files, use the `archive_id` value from `phase3v2.product_files`. The `access_url` column gives the direct download URL for each file, which can be used with any download tool. 

In [12]:
eso.retrieve_data(muse_previews["archive_id"][0], destination="./data/")

INFO: Downloading datasets ... [astroquery.eso.core]
INFO: Downloading 1 files ... [astroquery.eso.core]
INFO: Downloading file 1/1 https://dataportal.eso.org/dataPortal/file/ADP.2016-06-01T14:07:30.137 to /Users/abarnes/Library/CloudStorage/Dropbox/GitHub/astroquery_examples_eso/examples/advanced/data [astroquery.eso.core]
INFO: Found cached file /Users/abarnes/Library/CloudStorage/Dropbox/GitHub/astroquery_examples_eso/examples/advanced/data/ADP.2016-06-01T14:07:30.137.png [astroquery.eso.core]
INFO: Done! [astroquery.eso.core]


'/Users/abarnes/Library/CloudStorage/Dropbox/GitHub/astroquery_examples_eso/examples/advanced/data/ADP.2016-06-01T14:07:30.137.png'

<hr style="border:2px solid #0281c9"> </hr>